In [ ]:
## 5- Model Deploy Config
#### 5.1- Main Class

import pickle
import pandas as pd
import numpy as np
from category_encoders import OneHotEncoder, TargetEncoder


class HealthInsurance:

    def __init__( self ):
        self.home_path  = '' # Address
        self.encoders   = pickle.load( open( self.home_path + 'features/feature_encoders.pkl', 'rb' ) )


    def columns_rename(self, dataset):

        # Modificação nomes colunas
        new_columns = ['id', 'gender', 'age', 'driving_license', 'region_code', 'previously_insured', 
       'vehicle_age', 'vehicle_damage', 'annual_premium','policy_sales_channel', 'vintage']

        dataset.columns = new_columns

        return dataset

    def features_preparation(self, dataset):
        
        # Creating new feature
        dataset['age_group'] = pd.cut(dataset['age'], bins=13, labels=False)

        # Selected Features
        selected_features = ['age', 'region_code', 'previously_insured', 'vehicle_damage',
        'annual_premium', 'policy_sales_channel', 'age_group']

        # Encoding Features
        X = dataset[selected_features]

        test_data_ready = self.encoders.transform(X)

        return test_data_ready

    def get_prediction(self, model, original_data, test_data):

        prediction = model.predict_proba( test_data )

        original_data['score'] = prediction[:, 1].tolist()

        return original_data.to_json( orient='records', date_format='iso')


In [ ]:

### 5.2- API Handler
import pickle
# import os
import pandas as pd
from flask import Flask, request, Response, abort
from api.healthinsurance import HealthInsurance
from sklearn.ensemble import GradientBoostingClassifier

# loading model and encoders

model  = pickle.load( open( 'GB_model.pkl', 'rb' ) )


app = Flask( __name__)

@app.route('/')
def hello():
    return '<title>Machine Learning model working</title>'

@app.route( '/predict', methods=['POST'])

def health_insurance_predict():
    test_json = request.get_json()

    if test_json: # has data
        if isinstance( test_json, dict ): # unique example
            test_raw = pd.DataFrame( test_json, index=[0] )
        
        else: # multiple examples
            test_raw = pd.DataFrame( test_json, columns=test_json[0].keys() )
        
        # Instantiate Health Inssurance Class
        health_pipeline = HealthInsurance()

        # columns rename
        df1 = health_pipeline.columns_rename( test_raw )
        
        # Features transform
        df2 = health_pipeline.features_preparation( df1 )

        # prediction
        df_response = health_pipeline.get_prediction( model, test_raw, df2 )

        return df_response
    
    else:
        return Response( '{}', status=200, mimetype='aplication/json')
    
if __name__ == '__main__':
    # port = os.environ.get( 'PORT', 3000)
    app.run( host= '0.0.0.0', debug=True )


In [ ]:

### 5.2- API Tester
import requests
import json
import pandas as pd

test_df = pd.read_csv('data/test.csv')
data = json.dumps( test_df.head(100).to_dict( orient='record' ) )


test_df.sample(20)
test_df.sample(20).to_csv('testdata.csv')
# API Call

url = 'http://0.0.0.0:3000/predict'

# url = 'https://insurance-cross-sell-api-j7dp.onrender.com/predict'

header = { 'Content-type': 'application/json' }

r = requests.post( url, data=data, headers=header )
print( f'Status Code {r.status_code}')
## Checking the Data Received with Ranked Propensity Score 

d1 = pd.DataFrame( r.json(), columns=r.json()[0].keys() )

d1.sort_values( 'score', ascending=False )